# Practical 3: Pixel-Level Spatial GeoAI and GPS Drone Scouting
### Hands-on Session: Trouble Spots, Strong Spots, Fertilizer Advice and Farmer-in-the-Loop Intelligence with Google Gemini

In this practical:
- Ingest spatial telemetry derived from the 30-meter pixel raster cube.
- Find both the **trouble spots** and the **strongest, healthiest spots** in every plot, with Plot IDs and GPS coordinates.
- Add **farmer-provided field information** that satellites cannot see (fertilizer history, variety, soil, budget).
- Generate a precision Field Scouting, Fertilizer and Ground Action Plan.
- Let the AI **ask the farmer for missing information** instead of guessing, then give refined advice once the farmer answers.

## Step 1: Import required libraries

In [1]:
import os
import sys
import json
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv
from google import genai

## Step 2: Initialize Google GenAI Client with your API key from .env

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("GEOAI_MODEL", "gemini-3.5-flash-lite")

client = genai.Client(api_key=gemini_api_key)
print("Google GenAI Client successfully initialized.")
print("Target Model:", model_name)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Google GenAI Client successfully initialized.
Target Model: gemini-3.5-flash-lite


## Step 3: Ingest pixel-level spatial anomaly telemetry
Loading pre-computed spatial anomalies for all plots: Plot IDs, problem sectors, and exact GPS coordinates of the worst spot.

In [3]:
df_scout = pd.read_parquet("data/rice_farm_scout_anomalies.parquet")
df_scout['healthy_pixel_percent'] = (100 - df_scout['bad_pixel_percent']).round(1)

print("Pixel-Level Spatial Anomaly Telemetry:")
display(df_scout)

Pixel-Level Spatial Anomaly Telemetry:


,plot_id,total_pixels,bad_pixel_count,bad_pixel_percent,problem_sector,worst_spot_latitude,worst_spot_longitude,healthy_pixel_percent
0,Plot_01,381,44,11.5,Southern-Eastern corner,7.88312,8.32974,88.5
1,Plot_05,257,36,14.0,Southern-Eastern corner,7.88929,8.33860,86.0
2,Plot_15,332,32,9.6,Southern-Western corner,7.88446,8.33215,90.4
3,Plot_02,366,29,7.9,Southern-Eastern corner,7.88420,8.33135,92.1
4,Plot_06,334,26,7.8,Southern-Eastern corner,7.89117,8.32195,92.2
5,Plot_12,269,23,8.6,Northern-Western corner,7.90136,8.32436,91.4
6,Plot_11,334,22,6.6,Northern-Western corner,7.90029,8.32275,93.4
7,Plot_09,359,20,5.6,Southern-Eastern corner,7.88983,8.33914,94.4
8,Plot_04,366,20,5.5,Southern-Eastern corner,7.88688,8.33511,94.5
9,Plot_07,375,19,5.1,Northern-Western corner,7.89761,8.31872,94.9


## Step 4: Find the strongest, healthiest spot in every plot
Precision farming is not only about fixing bad patches. We also locate where the crop is doing best, so the farmer can learn from those areas.

Using the peak-growth layer of the pixel Zarr cube and the plot boundaries, we find for each plot:
- The GPS location of its strongest pixel.
- How vigorous its canopy was at peak growth compared with the rest of the farm (NDVI).
- How much nitrogen its leaves had at peak growth compared with the rest of the farm (NDRE), a useful signal for fertilizer planning.

We also read the time series to work out the crop's current growth stage, because fertilizer advice depends on it.

In [4]:
import xarray as xr
import geopandas as gpd

pixel_ds = xr.open_zarr("data/rice_farm_pixel_cube.zarr")
plots = gpd.read_file("farm_boundary/farm_boundary.geojson")[['plot_id', 'geometry']]

# Pixels at peak growth, tagged with the plot they fall in
peak = pixel_ds.sel(time='Peak_Vegetative')[['NDVI', 'NDRE']].to_dataframe().reset_index().dropna(subset=['NDVI', 'NDRE'])
pts = gpd.GeoDataFrame(peak, geometry=gpd.points_from_xy(peak['lon'], peak['lat']), crs="EPSG:4326")
pts = gpd.sjoin(pts, plots, predicate='within')

best = pts.loc[pts.groupby('plot_id')['NDVI'].idxmax(), ['plot_id', 'lat', 'lon']]
best = best.rename(columns={'lat': 'best_spot_latitude', 'lon': 'best_spot_longitude'}).round(5)
plot_stats = pts.groupby('plot_id')[['NDVI', 'NDRE']].median().reset_index()

# Plain-language labels relative to the rest of the farm, so the AI does not quote decimals
plot_stats['crop_vigour_at_peak'] = pd.qcut(plot_stats['NDVI'], 3, labels=['weaker than farm average', 'about farm average', 'among the strongest on the farm'])
plot_stats['leaf_nitrogen_at_peak'] = pd.qcut(plot_stats['NDRE'], 3, labels=['low', 'moderate', 'good'])

plot_data = (df_scout.merge(best, on='plot_id').merge(plot_stats.drop(columns=['NDVI', 'NDRE']), on='plot_id')
             .rename(columns={'bad_pixel_percent': 'trouble_area_percent', 'healthy_pixel_percent': 'healthy_area_percent'})
             [['plot_id', 'healthy_area_percent', 'trouble_area_percent', 'crop_vigour_at_peak', 'leaf_nitrogen_at_peak',
               'problem_sector', 'worst_spot_latitude', 'worst_spot_longitude', 'best_spot_latitude', 'best_spot_longitude']]
             .sort_values('healthy_area_percent', ascending=False))

# Current crop stage from the farm-wide time series
df_ts = pd.read_parquet("data/rice_farm_timeseries.parquet")
farm_ndvi = df_ts.groupby('date')['NDVI'].mean()
past_peak = farm_ndvi.iloc[-1] < 0.8 * farm_ndvi.max()
crop_stage = {
    "peak_growth_date": farm_ndvi.idxmax().strftime('%Y-%m-%d'),
    "latest_observation_date": farm_ndvi.index[-1].strftime('%Y-%m-%d'),
    "current_stage": "past peak growth, crop is ripening and drying down" if past_peak else "still in active growth",
}

scout_data = plot_data.astype({'crop_vigour_at_peak': str, 'leaf_nitrogen_at_peak': str}).to_dict(orient='records')

print("Crop stage:", json.dumps(crop_stage, indent=2))
display(plot_data)

Crop stage: {
  "peak_growth_date": "2024-11-04",
  "latest_observation_date": "2024-12-14",
  "current_stage": "past peak growth, crop is ripening and drying down"
}


,plot_id,healthy_area_percent,trouble_area_percent,crop_vigour_at_peak,leaf_nitrogen_at_peak,problem_sector,worst_spot_latitude,worst_spot_longitude,best_spot_latitude,best_spot_longitude
16,Plot_16,98.0,2.0,about farm average,moderate,Southern-Western corner,7.88125,8.33565,7.88393,8.33618
15,Plot_10,97.9,2.1,about farm average,good,Southern-Eastern corner,7.89251,8.34290,7.89493,8.33995
14,Plot_17,97.2,2.8,among the strongest on the farm,moderate,Southern-Western corner,7.87964,8.33753,7.88473,8.34344
13,Plot_03,96.4,3.6,about farm average,moderate,Southern-Eastern corner,7.88581,8.33350,7.89010,8.33027
11,Plot_14,95.3,4.7,among the strongest on the farm,good,Northern-Western corner,7.90351,8.32732,7.90271,8.33001
10,Plot_13,94.9,5.1,among the strongest on the farm,good,Northern-Eastern corner,7.89975,8.33054,7.89680,8.33269
9,Plot_07,94.9,5.1,weaker than farm average,low,Northern-Western corner,7.89761,8.31872,7.89546,8.31980
8,Plot_04,94.5,5.5,among the strongest on the farm,good,Southern-Eastern corner,7.88688,8.33511,7.89171,8.33189
7,Plot_09,94.4,5.6,among the strongest on the farm,good,Southern-Eastern corner,7.88983,8.33914,7.89198,8.33914
12,Plot_08,94.3,5.7,weaker than farm average,low,Southern-Eastern corner,7.89198,8.32571,7.89788,8.31980


## Step 5: Add farmer-provided field information
Satellites see the crop canopy, water and vegetation stress. They cannot see what fertilizer was applied, which variety was planted, the soil type, or the farmer's budget. Correct fertilizer and management advice needs this ground truth.

Fill in what you know. Leave anything you do not know as `None`. The AI is instructed to **ask the farmer** for missing details it needs, rather than guess.

In [5]:
farm_inputs = {
    "crop": "Rice",
    "variety": None,                     # e.g. "FARO 44"
    "planting_date": None,               # e.g. "2024-06-20"
    "fertilizer_already_applied": None,  # e.g. "NPK 15-15-15 at 2 bags/ha at planting, Urea 1 bag/ha at tillering"
    "soil_type": None,                   # e.g. "clay loam"
    "irrigation": None,                  # e.g. "canal-fed, can drain plots individually"
    "budget_for_inputs": None,           # e.g. "about 150,000 naira for this season"
    "farmer_observations": None,         # e.g. "yellow leaves in Plot_05, rats near the southern edge"
}


def known_inputs():
    return {k: v for k, v in farm_inputs.items() if v is not None}


def missing_inputs():
    return [k for k, v in farm_inputs.items() if v is None]


print("Farmer has provided:", json.dumps(known_inputs(), indent=2))
print("Still unknown:", missing_inputs())

Farmer has provided: {
  "crop": "Rice"
}
Still unknown: ['variety', 'planting_date', 'fertilizer_already_applied', 'soil_type', 'irrigation', 'budget_for_inputs', 'farmer_observations']


## Step 6: Generate Live Field Scouting, Fertilizer and Ground Action Plan
Google Gemini translates pixel telemetry and farmer information into a practical plan covering both weak and strong areas, with exact Plot IDs and GPS waypoints.

In [6]:
ADVISOR_RULES = """Do not use technical math jargon or decimal index numbers. Talk about practical farm reality (stunted seedlings, water pools, weed patches, soil leveling, leaf colour).
Only use the satellite data and the information the farmer has provided. Never invent facts about the farm (such as fertilizer already applied, variety or soil type).
If correct advice depends on information listed as unknown, say clearly what you can already tell from the satellite data, then ask the farmer up to 3 short, specific questions under a heading "Questions for the Farmer", explaining in one line why each answer matters.
Do not use emojis."""

prompt = f"""You are an expert Chief Farm Agronomist briefing a {farm_inputs['crop'].lower()} farmer and drone scout on their field today.
{ADVISOR_RULES}

Crop stage from satellite time series:
{json.dumps(crop_stage, indent=2)}

Plot-by-plot satellite telemetry (healthy and trouble areas, crop vigour and leaf nitrogen at peak growth, GPS of worst and best spots):
{json.dumps(scout_data, indent=2)}

Information provided by the farmer:
{json.dumps(known_inputs(), indent=2)}
Unknown so far: {missing_inputs()}

Write a direct Field Scouting and Action Plan:
1. Priority Plots with Trouble Spots: Name the top 3 plots with trouble spots (Plot ID and which corner or sector), what the scout is likely to see on the ground, and the GPS waypoint.
2. Best Performing Plots: Name the top 3 healthiest plots and give the GPS of their strongest spot. Explain what is likely going right there and what the farmer should check on the ground so the same practice can be copied to the weak plots.
3. Fertilizer Plan: Based on the crop stage and leaf nitrogen per plot, say plainly whether fertilizer is worth applying now. If yes, which plots, what type and roughly how much; if no, say why and what to plan for next season (which plots need more, which need less).
4. Immediate Ground Remedy: Simple instructions for workers at the trouble spots (e.g. dig a small channel to drain trapped water, spot-spray weeds).
5. Questions for the Farmer: Only if information you need is missing.

Format in clean Markdown with clear headings and bold Plot IDs."""

print("Calling Google Gemini GeoAI API for spatial scout report...")
response = client.models.generate_content(model=model_name, contents=prompt)
display(Markdown(response.text))

Calling Google Gemini GeoAI API for spatial scout report...


## Field Scouting and Action Plan

### 1. Priority Plots with Trouble Spots
Our satellite tracking shows that the farm has reached peak growth and is now entering the ripening and drying-down phase. However, a few plots carry notable trouble areas that the drone scout must inspect immediately on the ground.

* **Plot 05** (Southern-Eastern corner): The scout will likely see yellowing leaf color, stunted growth, or uneven ripening due to the highest trouble area on the farm. 
  * *Worst Spot GPS:* Latitude 7.88929, Longitude 8.33860
* **Plot 01** (Southern-Eastern corner): Expect to find patchy areas with poor canopy cover and delayed drying down compared to the rest of the field.
  * *Worst Spot GPS:* Latitude 7.88312, Longitude 8.32974
* **Plot 15** (Southern-Western corner): The scout should check for localized stunting, thin crop stands, or weed patches hiding beneath the canopy.
  * *Worst Spot GPS:* Latitude 7.88446, Longitude 8.33215

### 2. Best Performing Plots
To understand what success looks like right now, look to the plots that maintained strong crop vigour and good leaf nitrogen levels during peak growth.

* **Plot 13** (Strongest sector, Northern-Eastern area): 
  * *Best Spot GPS:* Latitude 7.89680, Longitude 8.33269
* **Plot 14** (Strongest sector, Northern-Western area): 
  * *Best Spot GPS:* Latitude 7.90271, Longitude 8.33001
* **Plot 09** (Strongest sector, Southern-Eastern area): 
  * *Best Spot GPS:* Latitude 7.89198, Longitude 8.33914

**What is going right:** These plots showed strong vigour and good leaf nitrogen at peak growth, meaning the crop established a healthy canopy and utilized nutrients efficiently. 
**What the farmer should check:** Walk these best-performing spots to check soil leveling, water retention, and uniform drainage. See if the good performance came from better land grading or cleaner weeding, so those exact practices can be applied to weak plots next season.

### 3. Fertilizer Plan
**Is fertilizer worth applying now?** No. 

Because the crop is past peak growth and is currently ripening and drying down, applying fertilizer now is a waste of money. The plants are no longer taking up nutrients for vegetative growth or grain filling in a way that will benefit yield. 

**Plan for next season:** 
* Plots that showed low leaf nitrogen at peak growth (**Plot 01, Plot 02, Plot 06, Plot 07, Plot 08, Plot 11**) will need a higher rate or better-timed split application of nitrogen next season.
* Plots that maintained good leaf nitrogen (**Plot 04, Plot 08, Plot 09, Plot 12, Plot 13, Plot 14**) received adequate nutrition and can maintain a similar fertilizer program next year.

### 4. Immediate Ground Remedy
Send workers to the trouble sectors on **Plot 05**, **Plot 01**, and **Plot 15** to perform these quick manual tasks:
* **Drain trapped water:** Walk the low spots and dig shallow hand-trenches or clear blocked channels to let any standing pools drain off so the grain can dry down evenly.
* **Spot-weed:** Pull out any late-season weed patches struggling through the thin canopy in the trouble corners so they do not drop seeds for the next cycle.

### Questions for the Farmer
* **What is your planting date?** This helps determine the exact maturity timeline and when harvest should begin.
* **What fertilizer has already been applied this season?** Knowing past applications explains why certain plots showed low leaf nitrogen while others stayed strong.
* **What is your soil type across these plots?** Soil type dictates how well nutrients and water are retained in the weak versus strong sectors.

## Step 7: Define interactive Q&A helper for field queries
The helper always reads the latest `farm_inputs`, so when the farmer answers the AI's questions, the next answer improves.

In [7]:
def ask_scout_ai(question):
    p = f"""You are an experienced {farm_inputs['crop']} Farm Operations Expert talking to a practical commercial farmer.
Always mention the specific Plot IDs, physical sectors (like southern corner, eastern side), and GPS coordinates where appropriate.
{ADVISOR_RULES}

Crop stage: {json.dumps(crop_stage)}
Field telemetry: {json.dumps(scout_data)}
Information provided by the farmer: {json.dumps(known_inputs())}
Unknown so far: {missing_inputs()}

Farmer asks: "{question}"
Answer directly and practically in 2-3 short paragraphs, followed by "Questions for the Farmer" only if needed."""
    r = client.models.generate_content(model=model_name, contents=p)
    return r.text

print("ask_scout_ai function ready.")

ask_scout_ai function ready.


## Step 8: Trouble Spot Question 1: Finding the Worst Bad Patch
*Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?*

In [8]:
q1 = "Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?"
print("Farmer asks:", q1)
display(Markdown(ask_scout_ai(q1)))

Farmer asks: Which specific plot has the worst patch of sick or stunted crops, and which corner should my farm workers walk to?


The worst trouble spot on the entire farm is in Plot_05, where you are dealing with the largest distressed area at fourteen percent. Your farm workers need to walk directly to the southern-eastern corner of that plot, specifically around latitude 7.88929 and longitude 8.33860, to inspect those stunted or failing plants. 

Another area requiring immediate attention is Plot_01, which has over eleven percent of its area struggling. Send your crew to the southern-eastern corner of Plot_01 near latitude 7.88312 and longitude 8.32974 to check for pale leaf color, weed patches, or pooling water as the crop finishes drying down.

Questions for the Farmer
1. What variety of rice is planted across these plots? (Determines natural maturation and ripening timelines.)
2. When was the final irrigation applied before dry down? (Explains potential moisture stress in southern and eastern corners.)
3. Have you noticed any pest damage or lodging in the worst spots of Plot_05 and Plot_01? (Helps distinguish physical crop damage from nutrient deficiency.)

## Step 9: Trouble Spot Question 2: Exact Phone GPS Waypoints
*Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.*

In [9]:
q2 = "Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately."
print("Farmer asks:", q2)
display(Markdown(ask_scout_ai(q2)))

Farmer asks: Give me the exact phone GPS waypoints I can give to my tractor driver or field scout to find the weak spots immediately.


Looking at our rice crop across the fields, we are now past peak growth, and the plants are ripening and drying down nicely in most areas. However, we do have distinct trouble spots where the crop struggled more during the season, showing lower leaf nitrogen and weaker vigour compared to the rest of the farm. 

To help your field scout or tractor driver check these lagging areas immediately, here are the exact GPS coordinates for the worst spots across your plots. In Plot_01, where we saw the highest trouble area of about eleven percent, send them to the southern-eastern corner at latitude 7.88312, longitude 8.32974. For Plot_05, which has about fourteen percent trouble area, the worst spot is nearby in the southern-eastern corner at latitude 7.88929, longitude 8.33860. In Plot_15, look to the southern-western corner at latitude 7.88446, longitude 8.33215. For Plot_11, check the northern-western corner at latitude 7.90029, longitude 8.32275. Finally, in Plot_06, head to the southern-eastern corner at latitude 7.89117, longitude 8.32195 to inspect the uneven ripening and any lingering weed patches or water pooling issues.

Questions for the Farmer
1. What variety of rice are you growing? This helps determine the normal ripening duration and expected dry-down appearance.
2. What are your field observations regarding drainage in the trouble sectors? This explains if water pools are causing the delayed dry-down.
3. Have you noticed lodging or pest damage in these specific corners? This clarifies if the weaker areas are due to physical crop damage rather than just low vigour.

## Step 10: Good Area Question: Learning from the Best Plots
*Which plots are doing best and where exactly is the strongest crop? What should I check there so I can copy it to my weak plots?*

In [10]:
q3 = "Which plots are doing best and where exactly is the strongest crop? What should I check there so I can copy it to my weak plots?"
print("Farmer asks:", q3)
display(Markdown(ask_scout_ai(q3)))

Farmer asks: Which plots are doing best and where exactly is the strongest crop? What should I check there so I can copy it to my weak plots?


Your best-performing plots right now are Plot_14, Plot_13, Plot_09, Plot_04, Plot_12, and Plot_17, which showed some of the strongest crop vigour on the farm back at peak growth. If you want to see where this healthy growth is concentrated, look at the best spots in these fields. For instance, in Plot_14, the strongest crop is found around latitude 7.90271, longitude 8.33001 in the upper section. In Plot_13, check around latitude 7.8968, longitude 8.33269 on the eastern side. Plot_09 shows its best growth around latitude 7.89198, longitude 8.33914 in the southern-eastern corner, while Plot_04 peaks nearby around latitude 7.89171, longitude 8.33189. Over in Plot_12, head to latitude 7.89546, longitude 8.33001 in the northern sector, and in Plot_17, the top-performing area sits around latitude 7.88473, longitude 8.34344 on the eastern side.

When you walk these best spots, check the soil moisture levels, how well the field was leveled, and the general root health compared to your struggling areas like Plot_01 and Plot_05. Since the crop is currently ripening and drying down past peak growth, observing the canopy thickness and grain filling in these strong zones will show you what went right. 

Questions for the Farmer
1. What variety of rice is planted across these plots? This matters because different varieties have unique ripening timelines and yield potentials.
2. What was the exact planting date for the plots? This helps us determine if maturity rates are matching up across the farm.
3. What fertilizer applications were made earlier in the season? This tells us if specific nutrient management practices drove the higher vigour in these top plots.

## Step 11: Fertilizer Question: Where, What and How Much
*Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?*

With most farm details still unknown, expect the AI to give what it can from the satellite data and then ask the farmer for the missing details.

In [11]:
q4 = "Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?"
print("Farmer asks:", q4)
display(Markdown(ask_scout_ai(q4)))

Farmer asks: Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?


Looking at the satellite data, your rice crop is past peak growth and currently ripening and drying down across the fields. Because the crop is at this late stage, applying fertilizer now will not help grain fill and risks leaving unused nutrients in the field or causing uneven maturity. Instead of looking at what to apply right now, we need to focus on where the crop struggled so you can plan ahead for the next season. 

Plots like Plot_01 and Plot_05 had the largest trouble areas reaching over ten percent, particularly clustered in their Southern-Eastern corners around latitude 7.88312, longitude 8.32974 in Plot_01 and latitude 7.88929, longitude 8.33860 in Plot_05, where leaf nitrogen and overall vigour were low during peak growth. On the other hand, plots like Plot_13 and Plot_14 held strong vigour with good leaf nitrogen, though Plot_14 still showed a trouble spot in its Northern-Western corner around latitude 7.90351, longitude 8.32732. You can safely skip late-season inputs on these stronger performing plots, but you will want to investigate those low-nitrogen spots before the next planting cycle.

Questions for the Farmer:
1. What variety of rice did you plant across these plots? This helps determine baseline nutrient and management needs.
2. What fertilizer applications have you already made this season? This tells us if the low nitrogen readings reflect missing inputs or poor uptake.
3. How are you currently managing irrigation and drainage in the problem corners? This explains whether water pooling or dryness caused the weaker vigour areas.

## Step 12: Farmer Answers the AI and Gets Refined Advice
Fill in the answers to the AI's questions below (example answers are shown; replace them with your real farm details), then ask the fertilizer question again. The advice now combines satellite intelligence with ground truth from the farmer.

In [12]:
farm_inputs.update({
    "variety": "FARO 44",
    "planting_date": "2024-06-20",
    "fertilizer_already_applied": "NPK 15-15-15 at 2 bags/ha at planting, no top dressing yet",
    "soil_type": "clay loam",
    "budget_for_inputs": "about 150,000 naira for this season",
    "farmer_observations": "yellowing leaves in the south-eastern part of Plot_05",
})

print("Still unknown:", missing_inputs())
print("Farmer asks again:", q4)
display(Markdown(ask_scout_ai(q4)))

Still unknown: ['irrigation']
Farmer asks again: Which plots need fertilizer, what type and how much should I apply, and where exactly? Which plots can I skip to save money?


Looking at the satellite data, your rice crop passed its peak growth back in early November and is now ripening and drying down across the fields. Since you are this late in the season and the crop is drying down for harvest, putting down any new fertilizer now will not help the grain fill and risks leaving you with delayed maturity or lodging. Instead, let us focus your budget on managing the weaker spots before harvest. 

Your biggest trouble spots are in Plot_05, where you already noticed yellowing leaves in the southern-eastern corner around latitude 7.8929, longitude 8.3386, and Plot_01, which has about eleven percent trouble area in its southern-eastern corner near latitude 7.8831, longitude 8.3297. Other plots like Plot_06, Plot_02, Plot_08, and Plot_11 also showed lower leaf nitrogen and weaker vigour earlier in the season, particularly in their southern-eastern and northern-western corners. You can safely skip applying anything to strong-performing plots like Plot_13, Plot_14, and Plot_09, which showed good vigour and nitrogen levels at peak growth. 

Questions for the Farmer
1. Are you planning any water management changes like draining the fields completely before harvest? (This determines if remaining wet spots will dry out in time.)
2. Have you noticed any pest or disease damage in the yellowing patches of Plot_05 and Plot_01? (This helps confirm if the leaf colour is purely due to late-season nutrient rundown or an active infestation.)